## Analyse data 11_analyze_data.ipynb 
reading in the remaining CSVs and org_id 1

### Summary
* Data in 15-minute intervals
* 64 organisations
* org_id, obj_id, period_begin, period_interval, value_type, value, value_quality

### Notebook content
* Goal: read in energy_data.csv and split into multiple files
* 01 look at the dataset with scan
* 02 read the dataset with read
* 03 split the dataset into 64 CSVs by org_id

### Findings
* Dataset has 541,445,297 rows and 7 columns
* too large to load directly, so polars is used
* for further processing, the data was split by org_id
*

#### Open
* pivot tables per REC by date
* optional aggregated values per REC in total without obj_id
* daylight-saving time change
* missing values
* NaN values
* everything present over time
* 


* evaluation script for data quality and domain view from the energy-industry perspective
  * KPIs per REC
  * totals across the whole REC
  * extreme values
  * how optimal the REC is
* export data csv + png

preprocessing
* daylight-saving time change
* replace NaN values with mean for short gaps
* build, fetch, and prepare the EPEX spot price dataset
* fetch and prepare weather data
* feature engineering -> compute optimisation values, price-ratio linkage


optimisation algorithm
* optimised schedule


script
* metering data, optimal data + schedule -> newly computed energy data
* notebook comparing actual and target schedule




ideas
* fair water distribution


New points
* analyse extreme values
* data quality check
* data preprocessing

In [ ]:
# imports
import numpy as np
import pandas as pd
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import EEG_Funktionen as eeg

In [ ]:
# konfig
objects_csv = 'data/objects.csv'
org_obj_csv = 'data/org_obj.csv'
organizations_csv = 'data/organizations.csv'
org_3_csv = 'data/split_files/org_3.csv'

# 01 Read in data

In [ ]:
objects_df = pd.read_csv(objects_csv)
objects_df.head()

In [ ]:
org_obj_df = pd.read_csv(org_obj_csv)
org_obj_df.head()

In [ ]:
organizations_df = pd.read_csv(organizations_csv)
organizations_df.head()

In [ ]:
org_3_df = pd.read_csv(org_3_csv)
org_3_df.head()

# 02 Datenexploration

In [ ]:
# short info about the dataset
summary = pd.DataFrame({
        'dtype': org_3_df.dtypes,
        'missing': org_3_df.isna().sum(),
        'min': org_3_df.min(numeric_only=False),
        'max': org_3_df.max(numeric_only=False)
    })

print(summary)

In [ ]:
for col in ["obj_id", "period_interval", "value_type", "value_quality"]:
    unique_vals = org_3_df[col].unique()
    print(f"\n {col} — {len(unique_vals)} unique values:")
    print(unique_vals)

value_quality -> grid level
 * L1 - 
 * L2 - 
 * L3 - 


value_type
* CC comm coverage minimum of cp/wmc - how much of the consumed electricity was community electricity
* CP comm potential - how much the energy community could have covered
* WMC weighted measured consumption - how much was consumed at that point in time
* WMG weighted measured generation - how much was fed in
* WSG surplus generation - how much surplus there was

In [ ]:
plt.figure(figsize=(10, 6))
org_3_df["value"].hist(bins=50)
plt.title("Distribution of values in 'value'")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
org_3_df_nozero = org_3_df[org_3_df["value"] > 0]

In [ ]:
plt.figure(figsize=(10, 6))
org_3_df_nozero["value"].hist(bins=50)
plt.title("Distribution of values in 'value' (values > 0)")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Analyse gaps

In [ ]:
pivot_df = org_3_df.pivot_table(
    index=["period_begin", "org_id", "obj_id", "value_quality"],  # all grouping features
    columns="value_type",   # columns: CC, CP, WMC, WMG, WSG
    values="value",         # values: the measurements
    aggfunc="mean"          # in case there are multiple measurements per combination
).reset_index()

In [ ]:
pivot_df.head(-1)

In [ ]:
pivot_df["period_begin"] = pd.to_datetime(pivot_df["period_begin"])

In [ ]:
# result list
gaps = []

# group by obj_id
for obj_id, group in pivot_df.groupby("obj_id"):
    group = group.sort_values("period_begin")

    # time differences between consecutive timestamps
    diffs = group["period_begin"].diff().dropna()

    # largest gap
    max_gap = diffs.max()

    # if gaps are not constant → problem
    if len(diffs.unique()) > 1:
        gaps.append({
            "obj_id": obj_id,
            "timestamp_count": len(group),
            "max_gap": max_gap,
            "irregular_spacing": True
        })
    else:
        gaps.append({
            "obj_id": obj_id,
            "timestamp_count": len(group),
            "max_gap": max_gap,
            "irregular_spacing": False
        })

# save as DataFrame
gap_df = pd.DataFrame(gaps)
print(gap_df)
